# Sold Week 6

In [2]:
import pandas as pd

sold = pd.read_csv("../../IDX_Data/sold_cleaned_week4_5.csv")
print("Shape:", sold.shape)
sold.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_94867/2464178534.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: BuyerAgencyCompensationType, 5: latfilled, 6: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("../../IDX_Data/sold_cleaned_week4_5.csv")


Shape: (396693, 72)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,LotSizeSquareFeet,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,year_month,rate_30yr_fixed,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425,False,False,False
1,SanDiego,SanDiego,NaN,False,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425,False,False,False
2,SanDiego,SanDiego,NaN,False,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,NaN,NaN,NaN,NaN,NaN,2024-01,6.6425,False,False,False
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,1079166779,davidmartz@compass.com,2024-01-30,858000.0,...,13504.0,NaN,NaN,NaN,NaN,2024-01,6.6425,False,True,True
4,Southland,Southland,NaN,False,False,1890500.0,1075037759,karen.klein@theagencyre.com,2024-01-29,1890500.0,...,17873.0,NaN,NaN,NaN,NaN,2024-01,6.6425,False,False,False


In [3]:
date_cols = [
    'CloseDate',
    'PurchaseContractDate',
    'ListingContractDate'
]

for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')

In [4]:
# Price ratio
sold['price_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']
# Price per sq ft
sold['price_per_sqft'] = sold['ClosePrice'] / sold['LivingArea']

In [5]:
# Year-month
sold['year_month'] = sold['CloseDate'].dt.to_period('M')

# Listing → Contract
sold['listing_to_contract_days'] = (
    sold['PurchaseContractDate'] - sold['ListingContractDate']
).dt.days

# Contract → Close
sold['contract_to_close_days'] = (
    sold['CloseDate'] - sold['PurchaseContractDate']
).dt.days

In [6]:
sold[[
    'ClosePrice',
    'OriginalListPrice',
    'price_ratio',
    'price_per_sqft',
    'listing_to_contract_days',
    'contract_to_close_days',
    'year_month'
]].head()

,ClosePrice,OriginalListPrice,price_ratio,price_per_sqft,listing_to_contract_days,contract_to_close_days,year_month
0,240000.0,499000.0,0.480962,210.526316,777.0,65.0,2024-01
1,815000.0,759900.0,1.072510,412.867275,114.0,919.0,2024-01
2,810000.0,739900.0,1.094743,410.334347,255.0,778.0,2024-01
3,858000.0,NaN,NaN,430.075188,188.0,-188.0,2024-01
4,1890500.0,1890500.0,1.000000,591.891046,0.0,0.0,2024-01


In [7]:
# Remove unrealistic or invalid time values
sold = sold[
    (sold['listing_to_contract_days'] >= 0) &
    (sold['contract_to_close_days'] >= 0)
]

In [8]:
import numpy as np

# Fix infinite values in price_ratio
sold['price_ratio'] = sold['price_ratio'].replace([np.inf, -np.inf], np.nan)
sold = sold[sold['OriginalListPrice'] > 0]

In [9]:
county_summary = sold.groupby('CountyOrParish').agg({
    'ClosePrice': ['mean', 'median'],
    'price_ratio': 'mean',
    'DaysOnMarket': 'mean'
}).reset_index()

county_summary.head()

CountyOrParish    ClosePrice            price_ratio DaysOnMarket
                          mean     median        mean         mean
0        Alameda  1.309950e+06  1135000.0    1.203557    25.951233
1         Alpine  1.100000e+06  1100000.0    0.666667   231.000000
2         Amador  4.038471e+05   409068.0    0.882618   127.227273
3          Butte  4.333909e+05   399000.0    1.488925    52.037046
4      Calaveras  5.687094e+05   485000.0    0.918105    68.977011

In [15]:
# --- Clean invalid price_ratio values ---

# Step 1: Remove bad OriginalListPrice values (like 0 or 1)
before_rows = sold.shape[0]

sold = sold[sold['OriginalListPrice'] > 10000]

# Step 2: Remove extreme/unrealistic price ratios
sold = sold[sold['price_ratio'] < 5]

after_rows = sold.shape[0]

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", after_rows)

# Step 3: Check distribution again
print("\nUpdated price_ratio summary:")
print(sold['price_ratio'].describe())

Rows before cleaning: 395481
Rows after cleaning: 394847

Updated price_ratio summary:
count    3.948470e+05
mean     9.883383e-01
std      9.247073e-02
min      9.207366e-07
25%      9.534091e-01
50%      9.953917e-01
75%      1.019332e+00
max      4.298441e+00
Name: price_ratio, dtype: float64


In [16]:
county_summary.columns = [
    'CountyOrParish',
    'mean_price',
    'median_price',
    'avg_price_ratio',
    'avg_days_on_market'
]

county_summary.head()

,CountyOrParish,mean_price,median_price,avg_price_ratio,avg_days_on_market
0,Alameda,1.309950e+06,1135000.0,1.203557,25.951233
1,Alpine,1.100000e+06,1100000.0,0.666667,231.000000
2,Amador,4.038471e+05,409068.0,0.882618,127.227273
3,Butte,4.333909e+05,399000.0,1.488925,52.037046
4,Calaveras,5.687094e+05,485000.0,0.918105,68.977011


In [17]:
type_summary = sold.groupby('PropertyType').agg({
    'ClosePrice': 'median',
    'price_ratio': 'mean',
    'DaysOnMarket': 'mean'
}).reset_index()

type_summary

,PropertyType,ClosePrice,price_ratio,DaysOnMarket
0,Residential,820000.0,0.988338,37.394246


In [18]:
print("Any inf in price_ratio:", np.isinf(sold['price_ratio']).sum()) 

Any inf in price_ratio: 0


In [19]:
sold.sort_values('price_ratio', ascending=False)[[
    'ClosePrice',
    'OriginalListPrice',
    'price_ratio'
]].head(10)

,ClosePrice,OriginalListPrice,price_ratio
146939,3860000.0,898000.0,4.298441
342596,2820000.0,705000.0,4.000000
342595,2820000.0,705000.0,4.000000
342594,2820000.0,705000.0,4.000000
342597,2820000.0,705000.0,4.000000
16020,130000.0,35000.0,3.714286
72446,1379000.0,374000.0,3.687166
303428,367000.0,100000.0,3.670000
169205,690000.0,189900.0,3.633491
251846,420000.0,120000.0,3.500000


In [20]:
sold.to_csv("sold_feature_engineered.csv", index=False)
print("Week 6 dataset saved.")

Week 6 dataset saved.


### Week 6 Summary

- Created key price and time features  
- Cleaned and validated new columns  
- Removed invalid values  
- Verified outputs with sample data  
- Generated grouped summaries for analysis  
- Prepared dataset for dashboard use